# RetinaSafe — Step 5: RETFound linear probe on BRSET

**Goal:** evaluate frozen RETFound CFP as a feature extractor for BRSET 12-label multilabel classification. Outputs the **first comparison condition** for the paper "Does retinal foundation model pretraining mitigate or amplify demographic biases?"

**Pipeline:**
1. Load frozen RETFound (ViT-L/16 CFP), no fine-tuning.
2. Extract CLS-token features (1024-dim) for all 16,266 BRSET images. Cache as `.npy`.
3. Train a single linear probe — `nn.Linear(1024, 12)` with BCE + per-label `pos_weight`.
4. Evaluate: per-label AUROC + bootstrap CIs, subgroup audit (sex / camera / quality / age), per-label calibration (ECE).
5. Compare numbers head-to-head with the ResNet-50 BRSET baseline you already have.

**Why linear probe (not fine-tuning):** isolates *what the pretraining knows* from *what the backbone can learn*. The full fine-tuning condition is a later step. Linear probe is the cleanest test of foundation model knowledge.

**Inputs to attach:**
1. `samarthmishra208/retfound-mae-nature-cfp-weights`
2. `samarthmishra208/brset-brazilian-multilabel-ophthalmological`
3. **Notebook Output:** `samarthmishra208/brset-baseline` (for the existing `brset_index.csv` — re-using the same patient-disjoint splits as the ResNet-50 baseline)

**Settings:** GPU T4 x2, Internet off. **Runtime ~30 min** (feature extraction is the bottleneck at ~25 min; everything else is fast).

## Cell 1 — Imports + locate inputs

In [ ]:
import os, sys, json, time, glob, pathlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score
import timm

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available(),
      "Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("timm:", timm.__version__)

# Find RETFound weights
wts = glob.glob("/kaggle/input/**/RETFound_mae_natureCFP.pth", recursive=True)
assert wts, "RETFound .pth not found"
WEIGHTS = wts[0]

# Find BRSET images
img_dirs = glob.glob("/kaggle/input/**/fundus_photos", recursive=True)
assert img_dirs, "BRSET fundus_photos not found"
IMAGES_DIR = img_dirs[0]

# Find brset_index.csv (from the BRSET baseline notebook output)
idx_csvs = glob.glob("/kaggle/input/**/brset_index.csv", recursive=True)
assert idx_csvs, "brset_index.csv not found — attach brset-baseline notebook output"
INDEX_CSV = idx_csvs[0]

print(f"\nWEIGHTS: {WEIGHTS}")
print(f"IMAGES_DIR: {IMAGES_DIR}")
print(f"INDEX_CSV: {INDEX_CSV}")

WORK = pathlib.Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
FEATURES_PATH = WORK / "retfound_features.npy"
FEATURES_META = WORK / "retfound_features_meta.csv"

DISEASE_COLS = [
    "diabetic_retinopathy", "macular_edema", "scar", "nevus", "amd",
    "vascular_occlusion", "hypertensive_retinopathy", "drusens",
    "hemorrhage", "myopic_fundus", "increased_cup_disc", "other",
]
N_LABELS = len(DISEASE_COLS)

## Cell 2 — Load frozen RETFound (ViT-L/16 CFP)

In [ ]:
raw = torch.load(WEIGHTS, map_location="cpu", weights_only=False)
state_dict = raw["model"] if isinstance(raw, dict) and "model" in raw else raw

backbone = timm.create_model("vit_large_patch16_224", pretrained=False,
                              num_classes=0, global_pool="")
missing, unexpected = backbone.load_state_dict(state_dict, strict=False)
print(f"RETFound load: missing={len(missing)}  unexpected={len(unexpected)}  (decoder keys expected as unexpected)")
assert len(missing) == 0, "Encoder did not fully load"

# Freeze
for p in backbone.parameters():
    p.requires_grad = False
backbone = backbone.to(DEVICE).eval()
print(f"Frozen RETFound on {DEVICE}. {sum(p.numel() for p in backbone.parameters())/1e6:.1f}M params (all frozen)")

## Cell 3 — Dataset + transforms (no augmentation for feature extraction)

Feature extraction should be deterministic — no augmentation. We extract features ONCE for every image and then re-use them for the linear probe (which is fast).

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

feat_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class BRSETAllSplitsDataset(Dataset):
    def __init__(self, index_csv, images_dir, transform=None):
        self.df = pd.read_csv(index_csv).reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.images_dir, f"{row['image_id']}.jpg")).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, idx  # return idx so we can write into the correct slot

ds = BRSETAllSplitsDataset(INDEX_CSV, IMAGES_DIR, transform=feat_tf)
print(f"Total images to encode: {len(ds):,}")

## Cell 4 — Extract RETFound CLS features

CLS token = first token of the 197-token sequence. 1024-dim per image. The ~25 min bottleneck of this notebook. Cached to disk so we never re-extract.

In [ ]:
if FEATURES_PATH.exists():
    features = np.load(FEATURES_PATH)
    print(f"Loaded cached features: {features.shape}")
else:
    BS = 64
    loader = DataLoader(ds, batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
    features = np.zeros((len(ds), 1024), dtype=np.float32)
    print(f"Extracting features for {len(ds):,} images...")
    t0 = time.time()
    n_done = 0
    with torch.no_grad():
        for x, idxs in loader:
            x = x.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                tokens = backbone.forward_features(x)   # [B, 197, 1024]
                cls = tokens[:, 0]                       # [B, 1024]
            features[idxs.numpy()] = cls.float().cpu().numpy()
            n_done += x.size(0)
            if n_done % 1024 == 0:
                rate = n_done / (time.time() - t0)
                eta = (len(ds) - n_done) / rate
                print(f"  {n_done:5d}/{len(ds):5d}  ({rate:.0f} img/s, ETA {eta/60:.1f} min)")
    np.save(FEATURES_PATH, features)
    print(f"Total: {(time.time()-t0)/60:.1f} min. Saved to {FEATURES_PATH}")

print(f"Features shape: {features.shape}  dtype: {features.dtype}")
print(f"Per-feature stats: mean={features.mean():.3f}  std={features.std():.3f}  "
      f"range=[{features.min():.2f}, {features.max():.2f}]")

## Cell 5 — Split features by train/val/test

Reuse the same patient-disjoint splits from the ResNet-50 baseline so head-to-head comparison is apples-to-apples.

In [ ]:
df = pd.read_csv(INDEX_CSV).reset_index(drop=True)
labels = df[DISEASE_COLS].astype(np.float32).to_numpy()  # (16266, 12)

train_mask = (df["split"] == "train").to_numpy()
val_mask   = (df["split"] == "val").to_numpy()
test_mask  = (df["split"] == "test").to_numpy()

X_train, y_train = features[train_mask], labels[train_mask]
X_val,   y_val   = features[val_mask],   labels[val_mask]
X_test,  y_test  = features[test_mask],  labels[test_mask]

print(f"train: X={X_train.shape}  y={y_train.shape}")
print(f"val:   X={X_val.shape}    y={y_val.shape}")
print(f"test:  X={X_test.shape}   y={y_test.shape}")

# Demographics for the test split (for the fairness audit)
test_df = df[test_mask].reset_index(drop=True)
test_sex = test_df["patient_sex"].astype(int).to_numpy()
test_cam = test_df["camera"].astype(str).to_numpy()
test_qual = test_df["quality"].astype(str).to_numpy()
test_age = test_df["patient_age"].apply(lambda x: float(x) if pd.notna(x) else -1.0).to_numpy()
print(f"\ntest demographics captured (n={len(test_df)})")

## Cell 6 — Train linear probe

Single `nn.Linear(1024, 12)`. BCE-with-logits, per-label `pos_weight`. AdamW. Quick — fits in tens of seconds.

In [ ]:
# Per-label pos_weight from TRAIN labels (same formula as ResNet-50 baseline)
n_pos = y_train.sum(axis=0)
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor(n_neg / np.maximum(n_pos, 1), dtype=torch.float32).to(DEVICE)

probe = nn.Linear(1024, N_LABELS).to(DEVICE)
optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32).to(DEVICE)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).to(DEVICE)

EPOCHS = 100
BS = 256
PATIENCE = 10
best_val_auroc = -1.0; best_state = None; no_improve = 0
n_train = len(X_train_t)

for epoch in range(EPOCHS):
    probe.train()
    perm = torch.randperm(n_train)
    losses = []
    for i in range(0, n_train, BS):
        idx = perm[i:i+BS]
        logits = probe(X_train_t[idx])
        loss = criterion(logits, y_train_t[idx])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
    train_loss = float(np.mean(losses))

    probe.eval()
    with torch.no_grad():
        val_logits = probe(X_val_t)
        val_loss = criterion(val_logits, y_val_t).item()
        val_probs = torch.sigmoid(val_logits).cpu().numpy()
    val_y = y_val_t.cpu().numpy()
    aurocs = []
    for i in range(N_LABELS):
        if 0 < val_y[:, i].sum() < len(val_y):
            aurocs.append(roc_auc_score(val_y[:, i], val_probs[:, i]))
    val_macro = float(np.mean(aurocs))
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_macro_auroc={val_macro:.4f}")
    if val_macro > best_val_auroc:
        best_val_auroc = val_macro
        best_state = {k: v.clone() for k, v in probe.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stop at epoch {epoch+1}")
            break

probe.load_state_dict(best_state)
print(f"\nBest val_macro_auroc = {best_val_auroc:.4f}")

## Cell 7 — Test eval: per-label AUROC with bootstrap 95% CIs

In [ ]:
probe.eval()
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    test_logits = probe(X_test_t)
    test_probs = torch.sigmoid(test_logits).cpu().numpy()

def bootstrap_auroc(p, y, n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    aurocs, auprcs = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yi, pi = y[idx], p[idx]
        if 0 < yi.sum() < len(yi):
            aurocs.append(roc_auc_score(yi, pi))
            auprcs.append(average_precision_score(yi, pi))
    def ci(v):
        v = np.asarray(v)
        return [float(np.quantile(v, alpha/2)), float(np.quantile(v, 1-alpha/2))]
    return dict(
        n=int(n), n_pos=int(y.sum()),
        auroc=float(np.mean(aurocs)) if aurocs else None,
        auroc_ci95=ci(aurocs) if aurocs else None,
        auprc=float(np.mean(auprcs)) if auprcs else None,
    )

per_label_results = {}
aurocs_overall = []
print("\n=== Per-label AUROC (RETFound linear probe on BRSET) ===")
for i, d in enumerate(DISEASE_COLS):
    r = bootstrap_auroc(test_probs[:, i], y_test[:, i])
    per_label_results[d] = r
    print(f"  {d:30s}  n_pos={r['n_pos']:4d}  AUROC={r['auroc']:.4f}  CI={r['auroc_ci95']}  AUPRC={r['auprc']:.4f}")
    aurocs_overall.append(r["auroc"])

macro_auroc = float(np.mean(aurocs_overall))
print(f"\nMacro AUROC: {macro_auroc:.4f}")

## Cell 8 — Subgroup fairness audit (with bootstrap CIs)

In [ ]:
MIN_N = 30; MIN_POS = 10

def audit_subgroup(probs, y_true, mask, name):
    if mask.sum() < MIN_N: return None
    p, y = probs[mask], y_true[mask]
    out = {"subgroup": name, "n": int(mask.sum()), "per_label": {}}
    for i, d in enumerate(DISEASE_COLS):
        if y[:, i].sum() >= MIN_POS and y[:, i].sum() < len(y):
            out["per_label"][d] = bootstrap_auroc(p[:, i], y[:, i])
        else:
            out["per_label"][d] = None
    return out

audit = {}
audit["by_sex"] = [a for a in [
    audit_subgroup(test_probs, y_test, test_sex == 1, "sex=male"),
    audit_subgroup(test_probs, y_test, test_sex == 2, "sex=female"),
] if a]
audit["by_camera"] = [a for a in [
    audit_subgroup(test_probs, y_test, test_cam == c, f"camera={c}")
    for c in sorted(set(test_cam)) if c
] if a]
audit["by_quality"] = [a for a in [
    audit_subgroup(test_probs, y_test, test_qual == q, f"quality={q}")
    for q in sorted(set(test_qual)) if q
] if a]
known_age = test_age >= 0
audit["by_age_band"] = [a for a in [
    audit_subgroup(test_probs, y_test, known_age & (test_age >= lo) & (test_age < hi), f"age={name}")
    for (lo, hi, name) in [(0,40,"<40"),(40,60,"40-60"),(60,80,"60-80"),(80,200,"80+")]
] if a]

# Identify gaps where CIs don't overlap
def find_gaps(rows, axis):
    print(f"\n=== Subgroup gaps where 95% CIs don't overlap ({axis}) ===")
    found = False
    for d in DISEASE_COLS:
        results = []
        for r in rows:
            v = r["per_label"].get(d)
            if v: results.append((r["subgroup"], v["auroc"], v["auroc_ci95"]))
        if len(results) < 2: continue
        for i in range(len(results)):
            for j in range(i+1, len(results)):
                a, b = results[i], results[j]
                if a[2][1] < b[2][0] or b[2][1] < a[2][0]:
                    found = True
                    print(f"  {d:30s}  {a[0]}: {a[1]:.3f} {a[2]}  vs  {b[0]}: {b[1]:.3f} {b[2]}")
    if not found: print("  (no statistically significant gaps)")

find_gaps(audit["by_sex"], "sex")
find_gaps(audit["by_camera"], "camera")
find_gaps(audit["by_quality"], "quality")
find_gaps(audit["by_age_band"], "age band")

## Cell 9 — Calibration analysis (per-label ECE)

Expected Calibration Error: bin predictions into 10 confidence buckets, measure |average_confidence − average_accuracy| in each. Lower is better.

In [ ]:
def expected_calibration_error(probs, y_true, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0; total = len(y_true)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i+1]
        mask = (probs >= lo) & (probs < hi)
        if i == n_bins - 1: mask = (probs >= lo) & (probs <= hi)
        if mask.sum() == 0: continue
        bin_acc = float(y_true[mask].mean())
        bin_conf = float(probs[mask].mean())
        ece += (mask.sum() / total) * abs(bin_acc - bin_conf)
    return float(ece)

per_label_ece = {}
print("\n=== Per-label Expected Calibration Error (ECE) ===")
for i, d in enumerate(DISEASE_COLS):
    e = expected_calibration_error(test_probs[:, i], y_test[:, i])
    per_label_ece[d] = e
    print(f"  {d:30s}  ECE={e:.4f}")

# Macro ECE
macro_ece = float(np.mean(list(per_label_ece.values())))
print(f"\nMacro ECE: {macro_ece:.4f}")

## Cell 10 — Persist artifacts

In [ ]:
results = {
    "method": "frozen_retfound_cfp_linear_probe",
    "best_val_macro_auroc": float(best_val_auroc),
    "test_n": int(len(y_test)),
    "test_macro_auroc": macro_auroc,
    "per_label": per_label_results,
    "fairness_audit": audit,
    "per_label_ece": per_label_ece,
    "macro_ece": macro_ece,
    "labels": DISEASE_COLS,
}
with open(RESULTS / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved test_metrics.json")

# Save probe weights (small — 1024*12 = 12k params)
torch.save(probe.state_dict(), RESULTS / "linear_probe.pt")

import shutil
zip_path = WORK / "retfound_probe_results.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(RESULTS))
print(f"Wrote {zip_path}  ({zip_path.stat().st_size/1e3:.1f} KB)")